# 056-De Bruijn Graph  （デュブラン）

de Bruijn graphはショートリードを用いたゲノムアセンブリに使われるデータ構造です。

リードをk-merに分割し、各k-merの前方k-1文字と、後方k-1文字を結ぶ辺として表します。  
辺を1回ずつ通る （オイラー路）と、その結果がゲノムアセンブリとなります。  
オイラー路は多項式時間で求められるため、de Bruijn graphを用いたアセンブリは、計算上扱いやすい問題となっています。

今回の問題は、各k-merの前方k-1文字と、後方k-1文字を求める問題です。


オランダ人数学者のNicolaas Govert de Bruijn （1918-2012）によって構築されました。
https://en.wikipedia.org/wiki/Nicolaas_Govert_de_Bruijn



In [1]:
reads = """TGAT
CATG
TCAT
ATGC
CATC
CATC""".splitlines()
print(reads)

['TGAT', 'CATG', 'TCAT', 'ATGC', 'CATC', 'CATC']


In [ ]:
def reverse_complement(s: str) -> str:
    comp = {'A': 'T', 'T': 'A', 'C': 'G', 'G': 'C'}
    return ''.join(comp[c] for c in reversed(s))

# 重複を消す
reads = set(reads)

# reverse complementの集合
reads_revcomp = {reverse_complement(read) for read in reads}

# すべてのリードの集合
all_reads = reads | reads_revcomp

# 各(k+1)-merから辺（k-mer）を作る。
# 入力として与えられる各文字列の長さが(k+1)なので、r[:-1]とr[1:]はそれぞれk-merになる。
edges = [(r[:-1], r[1:]) for r in all_reads]

# 出力
for u, v in edges:
    print(f"({u}, {v})")

(ATG, TGC)
(ATC, TCA)
(GCA, CAT)
(TGA, GAT)
(CAT, ATC)
(CAT, ATG)
(GAT, ATG)
(ATG, TGA)
(TCA, CAT)


## 本番

In [2]:
reads = """...
""".splitlines()
print(reads[:3])

['TGTCGTCTCAGGTGTTACCTCGCTCTTGGTGTCTTACTCTGCGTTATATT', 'ATCAGCTCGAGGGAGCGAGTAGACGCCGATCTCGTCCACCAATATAACGC', 'AATATAACGCAGAGTAAGACAACAAGAGCGAGGTAACACCTGAGACGACC']


In [ ]:
def reverse_complement(s: str) -> str:
    comp = {'A': 'T', 'T': 'A', 'C': 'G', 'G': 'C'}
    return ''.join(comp[c] for c in reversed(s))

reads = set(reads)
reads_revcomp = {reverse_complement(read) for read in reads}
all_reads = reads | reads_revcomp

edges = [(r[:-1], r[1:]) for r in all_reads]

for u, v in edges:
    print(f"({u}, {v})")

(...)
